In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Euclidean Distance threshold

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Otsu's method to determine threshold
def otsu_intraclass_variance(image, threshold):
  # Handle cases where `image` might contain NaNs or be empty for robust calculation
  valid_data = image[~np.isnan(image)]
  if len(valid_data) == 0:
      return np.inf # Return infinity if no valid data to prevent errors

  cls1 = valid_data[valid_data >= threshold]
  cls2 = valid_data[valid_data < threshold]

  # Avoid division by zero if a class is empty
  var1 = np.var(cls1) if len(cls1) > 0 else 0
  var2 = np.var(cls2) if len(cls2) > 0 else 0

  # Calculate weights based on the number of elements in each class
  weight1 = len(cls1) / len(valid_data)
  weight2 = len(cls2) / len(valid_data)

  return weight1 * var1 + weight2 * var2

# Load pre-compiled cohort dataset with all six features across all trials and sessions
excel_path = '/content/drive/MyDrive/<DATA_ROOT>/combined_feature_dataset.xlsx'
df = pd.read_excel(excel_path)
euc_dist_min = df['min_euclid_dist']
euc_dist_max = df['max_euclid_dist']


# Prepare data for Otsu's method, dropping NaN values
data_for_otsu = euc_dist_min.dropna().values  # threshold determined by euc dist min values due to binomial distribution

# Calculate min and max integer values for the threshold search range
# Ensure data_for_otsu is not empty
if len(data_for_otsu) > 0:
    min_val = int(np.floor(np.min(data_for_otsu)))
    max_val = int(np.ceil(np.max(data_for_otsu)))

    # Ensure min_val is less than max_val for range to work
    if min_val >= max_val:
        # If all values are the same or range is invalid, threshold might be the value itself
        otsu_threshold_ed = min_val
    else:
        # Find the threshold that minimizes the intraclass variance
        otsu_threshold_ed = min(
            range(min_val, max_val + 1), # +1 to include max_val in the range search
            key=lambda th: otsu_intraclass_variance(data_for_otsu, th),
        )
    print(f"Calculated Otsu Threshold: {otsu_threshold_ed}")
else:
    print("No valid data to calculate Otsu threshold.")
    otsu_threshold_ed = None # No threshold if data is empty

# Histogram
euc_dist_max.hist(bins = 60, edgecolor = 'black', color = 'orange', alpha = 0.5, label = 'Euclidean Distance (max)')
euc_dist_min.hist(bins = 70, edgecolor = 'black', alpha = 0.5, label = 'Euclidean Distance (min)')
plt.xlabel('Euclidean Distance (px)', fontsize = 16)
plt.ylabel('Frequency', fontsize = 16)
plt.tick_params(axis = 'both', labelsize = 16)
plt.axvline(otsu_threshold_ed, color = 'red', label = f' Otsu threshold: {otsu_threshold_ed}')  # vertical line indicating threshold value
plt.legend()

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/euc_dist_otsu_threshold.pdf', format='pdf', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/euc_dist_otsu_threshold.png', format='png', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/euc_dist_otsu_threshold.tiff', format='TIFF', bbox_inches='tight')

plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')
import scipy.stats as stats

# === CONFIGURATION ===
ctr_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID].csv']  # control subject trials
opto_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID].csv'] # opto subject trials

ctr_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']    # control manual scores
opto_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']   # opto manual scores

# === FUNCTIONS ===
# defining helper functions outside of main loop

def euclidean(x1, y1, x2, y2):
      return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

def classify_blink(csv_file, threshold):
    try:
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
        df = df.dropna()

        top_x, top_y = df['top_x'], df['top_y']
        bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
        euclid_dist = euclidean(top_x, top_y, bottom_x, bottom_y)

        blink_detected = (euclid_dist < threshold).any()
        return 1 if blink_detected else 0
    except:
        return None  # If any file fails, ignore it for now


# list storing accuracies for each individual trial/folder

ctr_accuracies_per_trial = []
opto_accuracies_per_trial = []

# loop going through each folder_path and its corresponding excel_path for CONTROL
for ctr_trial_indx, (ctr_current_folder_path, ctr_current_excel_path) in enumerate(zip(ctr_folder_path, ctr_excel_path)):

  print(f"\n--- Processing Trial {ctr_trial_indx + 1}: {ctr_current_folder_path} ---")

  # === LOAD GROUND TRUTH for current trial ===
  # using current_excel_path (a string) instead of the full excel_path list (to access the corresponding file)
  df_excel = pd.read_excel(ctr_current_excel_path, header=None)
  actual_all = df_excel[0].astype(int)  # grabbing the manually scored (actual) blink classification value

  # === SORT FILES for the current trial ===
  # using current_folder_path (string)
  ctr_sorted_files = sorted(
      [f for f in os.listdir(ctr_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # === HEADER ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store accuracies for the current trial
  ctr_current_trial_accuracies = []

  # === LOOP OVER THRESHOLDS CONTROL ===
  for threshold in range(0, 41):
      ctr_predicted = []
      ctr_valid_indices = []

      for i, filename in enumerate(ctr_sorted_files):
          ctr_file_path = os.path.join(ctr_current_folder_path, filename)
          blink_flag = classify_blink(ctr_file_path, threshold)

          if blink_flag is not None:
              ctr_predicted.append(blink_flag)
              ctr_valid_indices.append(i)

      # making sure actual and predicted values align after filtering out valid_indices
      ctr_actual = actual_all.iloc[ctr_valid_indices].reset_index(drop=True)
      ctr_predicted = pd.Series(ctr_predicted)

      # Filter out actual == 3
      mask = ctr_actual != 3
      ctr_actual_filtered = ctr_actual[mask].reset_index(drop=True)
      ctr_predicted_filtered = ctr_predicted[mask].reset_index(drop=True)

      # Accuracy logic
      ctr_correct = (ctr_actual_filtered == ctr_predicted_filtered) | ((ctr_actual_filtered == 2) & (ctr_predicted_filtered == 0))
      ctr_accuracy = ctr_correct.sum() / len(ctr_correct) * 100 if len(ctr_correct) > 0 else 0
      ctr_total_blinks = ctr_predicted_filtered.sum()

      ctr_current_trial_accuracies.append(ctr_accuracy)  # Adding accuracy for this threshold to the current trials' list

      # Print tab-separated (copy-paste friendly)
      print(f"{threshold}\t{ctr_accuracy:.2f}\t{ctr_total_blinks}")

  ctr_accuracies_per_trial.append(ctr_current_trial_accuracies) # Adding the list of accuracies for the current trial to the master list


# loop going through each folder_path and its corresponding excel_path for OPTO
for opto_trial_indx, (opto_current_folder_path, opto_current_excel_path) in enumerate(zip(opto_folder_path, opto_excel_path)):

  print(f"\n--- Processing Trial {opto_trial_indx + 1}: {opto_current_folder_path} ---")

  # === LOAD GROUND TRUTH for current trial ===
  # using current_excel_path (a string) instead of the full excel_path list (to access the corresponding file)
  df_excel = pd.read_excel(opto_current_excel_path, header=None)
  actual_all = df_excel[0].astype(int)  # grabbing the manually scored (actual) blink classification value

  # === SORT FILES for the current trial ===
  # using current_folder_path (string)
  opto_sorted_files = sorted(
      [f for f in os.listdir(opto_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # === HEADER ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store accuracies for the current trial
  opto_current_trial_accuracies = []

  # === LOOP OVER THRESHOLDS OPTO ===
  for threshold in range(0, 41):
      opto_predicted = []
      opto_valid_indices = []

      for i, filename in enumerate(opto_sorted_files):
          opto_file_path = os.path.join(opto_current_folder_path, filename)
          blink_flag = classify_blink(opto_file_path, threshold)

          if blink_flag is not None:
              opto_predicted.append(blink_flag)
              opto_valid_indices.append(i)

      # making sure actual and predicted values align after filtering out valid_indices
      opto_actual = actual_all.iloc[opto_valid_indices].reset_index(drop=True)
      opto_predicted = pd.Series(opto_predicted)

      opto_actual = actual_all.iloc[opto_valid_indices].reset_index(drop=True)
      opto_predicted = pd.Series(opto_predicted)

      # Filter out actual == 3
      mask = opto_actual != 3
      opto_actual_filtered = opto_actual[mask].reset_index(drop=True)
      opto_predicted_filtered = opto_predicted[mask].reset_index(drop=True)

      # Accuracy logic
      opto_correct = (opto_actual_filtered == opto_predicted_filtered) | ((opto_actual_filtered == 2) & (opto_predicted_filtered == 0))
      opto_accuracy = opto_correct.sum() / len(opto_correct) * 100 if len(opto_correct) > 0 else 0
      opto_total_blinks = opto_predicted_filtered.sum()

      opto_current_trial_accuracies.append(opto_accuracy)  # Adding accuracy for this threshold to the current trials' list

      # Print tab-separated (copy-paste friendly)
      print(f"{threshold}\t{opto_accuracy:.2f}\t{opto_total_blinks}")

  opto_accuracies_per_trial.append(opto_current_trial_accuracies) # Adding the list of accuracies for the current trial to the master list

# Calculating avg accuracy across trials (moved outside the trial loop)
ctr_avg_accuracy = []
opto_avg_accuracy = []
ctr_sems_per_threshold = []
opto_sems_per_threshold = [] # New list to store SEM for each threshold

from itertools import zip_longest

# fillvalue=None lets us identify and skip missing data points
for column in zip_longest(*ctr_accuracies_per_trial, fillvalue=None):
    ctr_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if ctr_valid_numbers:
      ctr_avg_accuracy.append(sum(ctr_valid_numbers) / len(ctr_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      ctr_sems_per_threshold.append(stats.sem(ctr_valid_numbers))
    else:
      ctr_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      ctr_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers


for column in zip_longest(*opto_accuracies_per_trial, fillvalue=None):
    opto_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if opto_valid_numbers:
      opto_avg_accuracy.append(sum(opto_valid_numbers) / len(opto_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      opto_sems_per_threshold.append(stats.sem(opto_valid_numbers))
    else:
      opto_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      opto_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers

# Convert lists to numpy arrays for element-wise operations
ctr_avg_accuracy_np = np.array(ctr_avg_accuracy)
ctr_sems_np = np.array(ctr_sems_per_threshold)
opto_avg_accuracy_np = np.array(opto_avg_accuracy)
opto_sems_np = np.array(opto_sems_per_threshold)

max_ctr_accuracy = np.max(ctr_avg_accuracy_np)
max_ctr_accuracy_threshold_index = np.argmax(ctr_avg_accuracy_np)
max_ctr_sem = ctr_sems_np[max_ctr_accuracy_threshold_index]

max_opto_accuracy = np.max(opto_avg_accuracy_np)
max_opto_accuracy_threshold_index = np.argmax(opto_avg_accuracy_np)
max_opto_sem = opto_sems_np[max_opto_accuracy_threshold_index]

# Combined max avg accuracy of experimental and control
opto_ctr_accuracy = []   # avg accuracy at each index of opto and ctr lists
# max_opto_ctr_accuracy = np.max(opto_ctr_accuracy)

for indx in range(len(ctr_avg_accuracy_np)):
  if abs(ctr_avg_accuracy_np[indx] - opto_avg_accuracy_np[indx]) < 2:
    opto_ctr_accuracy.append((ctr_avg_accuracy_np[indx] + opto_avg_accuracy_np[indx])/2)

  else:
    opto_ctr_accuracy.append(0)

# index of maximum avg accuracy of opto and ctr combined that are within 2% of each other
max_opto_ctr_accuracy_threshold_index = np.argmax(opto_ctr_accuracy) # index of max avg avvuracy value of opto and ctr list tgt

# Generating avg across trials graph
fig, ax = plt.subplots(figsize = (10, 6))
ax.plot(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np, label='Control air puff only', color = '#b80d07') # Use numpy array for plotting
ax.plot(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np, label='Opto air puff only', color = '#026bdb')
ax.set_xlabel('Euclidean Distance Threshold (px)', fontsize = 16)
ax.set_ylabel('Classification Accuracy (%)', fontsize = 16)
ax.set_title('Accuracy vs. Euclidean Distance Threshold for Control and Opto air puff only trials', fontsize = 16, fontweight = 'bold')
ax.grid(True, linestyle='--', alpha=0.7)
ax.tick_params(axis = 'both', labelsize = 16)
# SEM
# Use numpy arrays for fill_between and add alpha for better visualization
ax.fill_between(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np - ctr_sems_np, ctr_avg_accuracy_np + ctr_sems_np, alpha=0.2, label = 'ctr_SEM', color = '#b80d07')
ax.fill_between(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np - opto_sems_np, opto_avg_accuracy_np + opto_sems_np, alpha=0.2, label = 'opto_SEM', color = '#026bdb')

ax.axvline(x = max_opto_ctr_accuracy_threshold_index, color = 'black', label = 'Validated threshold')
ax.axvline(x= otsu_threshold_ed, color = 'black', linestyle = '--', label = 'Otsu threshold')

ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 2)

ax.legend(bbox_to_anchor = (0.5, -0.15), loc = 'upper center', ncol=6)

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_v_euc_dist_threshold.pdf', format='pdf', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_euc_dist_threshold.tiff', format='TIFF', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_euc_dist_threshold.png', format='png', bbox_inches = 'tight')

plt.show()

print(f"Control - Maximum Accuracy: {max_ctr_accuracy:.2f}% at threshold index {max_ctr_accuracy_threshold_index}, with SEM: {max_ctr_sem:.2f}")
print(f"Opto - Maximum Accuracy: {max_opto_accuracy:.2f}% at threshold index {max_opto_accuracy_threshold_index}, with SEM: {max_opto_sem:.2f}")
print(f"Control + Opto Threshold Index: {max_opto_ctr_accuracy_threshold_index}")
print(f"Ctr accuracy at overall threshold index: {ctr_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}")
print(f"Opto accuracy at overall threshold index:{opto_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}")


print(f'Otsu control avg accuracy: {ctr_avg_accuracy_np[otsu_threshold_ed]}')
print(f'Otsu opto avg accuracy: {opto_avg_accuracy_np[otsu_threshold_ed]}')

In [ ]:
# EAR threshold

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Otsu's method to determine threshold
def otsu_intraclass_variance(image, threshold):
  # Handle cases where `image` might contain NaNs or be empty for robust calculation
  valid_data = image[~np.isnan(image)]
  if len(valid_data) == 0:
      return np.inf # Return infinity if no valid data to prevent errors

  cls1 = valid_data[valid_data >= threshold]
  cls2 = valid_data[valid_data < threshold]

  # Avoid division by zero if a class is empty
  var1 = np.var(cls1) if len(cls1) > 0 else 0
  var2 = np.var(cls2) if len(cls2) > 0 else 0

  # Calculate weights based on the number of elements in each class
  weight1 = len(cls1) / len(valid_data)
  weight2 = len(cls2) / len(valid_data)

  return weight1 * var1 + weight2 * var2

# Load pre-compiled cohort dataset with all six features across all trials and sessions
excel_path = '/content/drive/MyDrive/<DATA_ROOT>/combined_feature_dataset.xlsx'
df = pd.read_excel(excel_path)
ear_min = df['min_ear']
ear_max = df['max_ear']


# Prepare data for Otsu's method, dropping NaN values
data_for_otsu = ear_min.dropna().values  # threshold determined by euc dist min values due to binomial distribution

# Calculate min and max integer values for the threshold search range
# Ensure data_for_otsu is not empty
if len(data_for_otsu) > 0:
    min_val = (np.min(data_for_otsu))
    max_val = (np.max(data_for_otsu))

    # Ensure min_val is less than max_val for range to work
    if min_val >= max_val:
        # If all values are the same or range is invalid, threshold might be the value itself
        otsu_threshold_ear = min_val
    else:
        # Find the threshold that minimizes the intraclass variance
        threshold_candidates = np.linspace(min_val, max_val, num=1000)
        otsu_threshold_ear = min(
            threshold_candidates,
            key=lambda th: otsu_intraclass_variance(data_for_otsu, th),
        )
    print(f"Calculated Otsu Threshold: {otsu_threshold_ear}")
else:
    print("No valid data to calculate Otsu threshold.")
    otsu_threshold_ear = None # No threshold if data is empty

# Histogram
ear_max.hist(bins = 60, edgecolor = 'black', color = 'orange', alpha = 0.5, label = 'EAR (max)')
ear_min.hist(bins = 70, edgecolor = 'black', alpha = 0.5, label = 'EAR (min)')
plt.xlabel('EAR (A.U.)', fontsize = 16)
plt.ylabel('Frequency', fontsize = 16)
plt.tick_params(axis = 'both', labelsize = 16)
plt.axvline(otsu_threshold_ear, color = 'red', label = f'Otsu threshold: {round(otsu_threshold_ear, 2)}')  # vertical line indicating threshold value
plt.legend()

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/ear_otsu_threshold.pdf', format='pdf', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/ear_otsu_threshold.png', format='png', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/ear_otsu_threshold.tiff', format='TIFF', bbox_inches = 'tight')

plt.show()

In [ ]:
# opto + ctr Eye Aspect Ratio (EAR) Figure with SEM

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')
import scipy.stats as stats

# === CONFIGURATION ===
ctr_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID].csv']  # control subject trials
opto_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID].csv']  # opto subject trials

ctr_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']    # control manual scores
opto_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']   # opto manual scores

# === FUNCTIONS ===
# defining helper functions outside of main loop

def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

def classify_blink_ear(csv_file, threshold):
    try:
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
        df = df.dropna()

        # Extract EAR points
        p1_x, p1_y = df['p1_x'], df['p1_y']
        p2_x, p2_y = df['p2_x'], df['p2_y']
        p3_x, p3_y = df['p3_x'], df['p3_y']
        p4_x, p4_y = df['p4_x'], df['p4_y']
        p5_x, p5_y = df['p5_x'], df['p5_y']
        p6_x, p6_y = df['p6_x'], df['p6_y']

        vertical1 = euclidean(p2_x, p2_y, p6_x, p6_y)
        vertical2 = euclidean(p3_x, p3_y, p5_x, p5_y)
        horizontal = euclidean(p1_x, p1_y, p4_x, p4_y)
        ear = (vertical1 + vertical2) / (2.0 * horizontal)

        blink_detected = (ear < threshold).any()
        return 1 if blink_detected else 0
    except:
        return None

# list storing accuracies for each individual trial/folder

ctr_accuracies_per_trial = []
opto_accuracies_per_trial = []

# loop going through each folder_path and its corresponding excel_path for CONTROL
for ctr_trial_indx, (ctr_current_folder_path, ctr_current_excel_path) in enumerate(zip(ctr_folder_path, ctr_excel_path)):

  print(f"\n--- Processing Trial {ctr_trial_indx + 1}: {ctr_current_folder_path} ---")

  # === LOAD GROUND TRUTH for current trial ===
  # using current_excel_path (a string) instead of the full excel_path list (to access the corresponding file)
  df_excel = pd.read_excel(ctr_current_excel_path, header=None)
  actual_all = df_excel[0].astype(int)  # grabbing the manually scored (actual) blink classification value

  # === SORT FILES for the current trial ===
  # using current_folder_path (string)
  ctr_sorted_files = sorted(
      [f for f in os.listdir(ctr_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # === HEADER ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store accuracies for the current trial
  ctr_current_trial_accuracies = []

  # === LOOP OVER THRESHOLDS CONTROL ===
  thresholds = np.round(np.arange(0.00, 1.01, 0.01), 2)

  for threshold in thresholds:
      ctr_predicted = []
      ctr_valid_indices = []

      for i, filename in enumerate(ctr_sorted_files):
          ctr_file_path = os.path.join(ctr_current_folder_path, filename)
          blink_flag = classify_blink_ear(ctr_file_path, threshold)

          if blink_flag is not None:
              ctr_predicted.append(blink_flag)
              ctr_valid_indices.append(i)

      # making sure actual and predicted values align after filtering out valid_indices
      ctr_actual = actual_all.iloc[ctr_valid_indices].reset_index(drop=True)
      ctr_predicted = pd.Series(ctr_predicted)

      # Filter out actual == 3
      mask = ctr_actual != 3
      ctr_actual_filtered = ctr_actual[mask].reset_index(drop=True)
      ctr_predicted_filtered = ctr_predicted[mask].reset_index(drop=True)

      # Accuracy logic
      ctr_correct = (ctr_actual_filtered == ctr_predicted_filtered) | ((ctr_actual_filtered == 2) & (ctr_predicted_filtered == 0))
      ctr_accuracy = ctr_correct.sum() / len(ctr_correct) * 100 if len(ctr_correct) > 0 else 0
      ctr_total_blinks = ctr_predicted_filtered.sum()

      ctr_current_trial_accuracies.append(ctr_accuracy)  # Adding accuracy for this threshold to the current trials' list

      # Print tab-separated (copy-paste friendly)
      print(f"{threshold}\t{ctr_accuracy:.2f}\t{ctr_total_blinks}")

  ctr_accuracies_per_trial.append(ctr_current_trial_accuracies) # Adding the list of accuracies for the current trial to the master list


# loop going through each folder_path and its corresponding excel_path for OPTO
for opto_trial_indx, (opto_current_folder_path, opto_current_excel_path) in enumerate(zip(opto_folder_path, opto_excel_path)):

  print(f"\n--- Processing Trial {opto_trial_indx + 1}: {opto_current_folder_path} ---")

  # === LOAD GROUND TRUTH for current trial ===
  # using current_excel_path (a string) instead of the full excel_path list (to access the corresponding file)
  df_excel = pd.read_excel(opto_current_excel_path, header=None)
  actual_all = df_excel[0].astype(int)  # grabbing the manually scored (actual) blink classification value

  # === SORT FILES for the current trial ===
  # using current_folder_path (string)
  opto_sorted_files = sorted(
      [f for f in os.listdir(opto_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # === HEADER ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store accuracies for the current trial
  opto_current_trial_accuracies = []


  thresholds = np.round(np.arange(0.00, 1.01, 0.01), 2)
  # === LOOP OVER THRESHOLDS OPTO ===
  for threshold in thresholds:
      opto_predicted = []
      opto_valid_indices = []

      for i, filename in enumerate(opto_sorted_files):
          opto_file_path = os.path.join(opto_current_folder_path, filename)
          blink_flag = classify_blink_ear(opto_file_path, threshold)

          if blink_flag is not None:
              opto_predicted.append(blink_flag)
              opto_valid_indices.append(i)

      # making sure actual and predicted values align after filtering out valid_indices
      opto_actual = actual_all.iloc[opto_valid_indices].reset_index(drop=True)
      opto_predicted = pd.Series(opto_predicted)

      opto_actual = actual_all.iloc[opto_valid_indices].reset_index(drop=True)
      opto_predicted = pd.Series(opto_predicted)

      # Filter out actual == 3
      mask = opto_actual != 3
      opto_actual_filtered = opto_actual[mask].reset_index(drop=True)
      opto_predicted_filtered = opto_predicted[mask].reset_index(drop=True)

      # Accuracy logic
      opto_correct = (opto_actual_filtered == opto_predicted_filtered) | ((opto_actual_filtered == 2) & (opto_predicted_filtered == 0))
      opto_accuracy = opto_correct.sum() / len(opto_correct) * 100 if len(opto_correct) > 0 else 0
      opto_total_blinks = opto_predicted_filtered.sum()

      opto_current_trial_accuracies.append(opto_accuracy)  # Adding accuracy for this threshold to the current trials' list

      # Print tab-separated (copy-paste friendly)
      print(f"{threshold}\t{opto_accuracy:.2f}\t{opto_total_blinks}")

  opto_accuracies_per_trial.append(opto_current_trial_accuracies) # Adding the list of accuracies for the current trial to the master list

# Calculating avg accuracy across trials (moved outside the trial loop)
ctr_avg_accuracy = []
opto_avg_accuracy = []
ctr_sems_per_threshold = []
opto_sems_per_threshold = [] # New list to store SEM for each threshold

from itertools import zip_longest

# fillvalue=None lets us identify and skip missing data points
for column in zip_longest(*ctr_accuracies_per_trial, fillvalue=None):
    ctr_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if ctr_valid_numbers:
      ctr_avg_accuracy.append(sum(ctr_valid_numbers) / len(ctr_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      ctr_sems_per_threshold.append(stats.sem(ctr_valid_numbers))
    else:
      ctr_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      ctr_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers


for column in zip_longest(*opto_accuracies_per_trial, fillvalue=None):
    opto_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if opto_valid_numbers:
      opto_avg_accuracy.append(sum(opto_valid_numbers) / len(opto_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      opto_sems_per_threshold.append(stats.sem(opto_valid_numbers))
    else:
      opto_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      opto_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers

# Convert lists to numpy arrays for element-wise operations
ctr_avg_accuracy_np = np.array(ctr_avg_accuracy)
ctr_sems_np = np.array(ctr_sems_per_threshold)
opto_avg_accuracy_np = np.array(opto_avg_accuracy)
opto_sems_np = np.array(opto_sems_per_threshold)

max_ctr_accuracy = np.max(ctr_avg_accuracy_np)
max_ctr_accuracy_threshold_index = np.argmax(ctr_avg_accuracy_np)
max_ctr_sem = ctr_sems_np[max_ctr_accuracy_threshold_index]

max_opto_accuracy = np.max(opto_avg_accuracy_np)
max_opto_accuracy_threshold_index = np.argmax(opto_avg_accuracy_np)
max_opto_sem = opto_sems_np[max_opto_accuracy_threshold_index]

# Combined max avg accuracy of experimental and control
opto_ctr_accuracy = []   # avg accuracy at each index of opto and ctr lists
# max_opto_ctr_accuracy = np.max(opto_ctr_accuracy)

for indx in range(len(ctr_avg_accuracy_np)):
  if abs(ctr_avg_accuracy_np[indx] - opto_avg_accuracy_np[indx]) < 2:
    opto_ctr_accuracy.append((ctr_avg_accuracy_np[indx] + opto_avg_accuracy_np[indx])/2)

  else:
    opto_ctr_accuracy.append(0)

# index of maximum avg accuracy of opto and ctr combined that are within 2% of each other
max_opto_ctr_accuracy_threshold_index = np.argmax(opto_ctr_accuracy) # index of max avg avvuracy value of opto and ctr list tgt


# Generating avg across trials graph
fig, ax = plt.subplots(figsize = (10, 6))
ax.plot(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np, label='Control', color = '#b80d07') # Use numpy array for plotting
ax.plot(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np, label='Opto', color = '#026bdb')
ax.set_xlabel('EAR Threshold (A.U.)', fontsize = 16)
ax.set_ylabel('Classification Accuracy (%)', fontsize = 16)
ax.set_title('Accuracy vs. EAR Threshold for Control and Opto', fontsize = 16, fontweight = 'bold')
ax.grid(True, linestyle='--', alpha=0.7)
ax.tick_params(axis = 'both', labelsize = 16)
# SEM
# Use numpy arrays for fill_between and add alpha for better visualization
ax.fill_between(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np - ctr_sems_np, ctr_avg_accuracy_np + ctr_sems_np, alpha=0.2, label = 'ctr_SEM', color = '#b80d07')
ax.fill_between(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np - opto_sems_np, opto_avg_accuracy_np + opto_sems_np, alpha=0.2, label = 'opto_SEM', color = '#026bdb')

ax.axvline(x = max_opto_ctr_accuracy_threshold_index, color = 'black', label = 'Validated threshold')
ax.axvline(x= (round(otsu_threshold_ear*100)), color = 'black', linestyle = '--', label = 'Otsu threshold')

ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 2)

ax.legend(bbox_to_anchor = (0.5, -0.15), loc = 'upper center', ncol=6)

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_v_ear_threshold.pdf', format='pdf', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_ear_threshold.tiff', format='TIFF', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_ear_threshold.png', format='png', bbox_inches = 'tight')

plt.show()

print(f"Control - Maximum Accuracy: {max_ctr_accuracy:.2f}% at threshold index {max_ctr_accuracy_threshold_index}, with SEM: {max_ctr_sem:.2f}")
print(f"Opto - Maximum Accuracy: {max_opto_accuracy:.2f}% at threshold index {max_opto_accuracy_threshold_index}, with SEM: {max_opto_sem:.2f}")
print(f"Control + Opto Threshold Index: {max_opto_ctr_accuracy_threshold_index}")
print(f'Control accuracy at overall threshold index: {ctr_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')
print(f'Opto accuracy at overall threshold index: {opto_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')

print(f'Otsu control avg accuracy: {ctr_avg_accuracy_np[round(otsu_threshold_ear*100)]}')
print(f'Otsu opto avg accuracy: {opto_avg_accuracy_np[round(otsu_threshold_ear*100)]}')

In [ ]:
# Inner Eye Angle threshold

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Otsu's method to determine threshold
def otsu_intraclass_variance(image, threshold):
  # Handle cases where `image` might contain NaNs or be empty for robust calculation
  valid_data = image[~np.isnan(image)]
  if len(valid_data) == 0:
      return np.inf # Return infinity if no valid data to prevent errors

  cls1 = valid_data[valid_data >= threshold]
  cls2 = valid_data[valid_data < threshold]

  # Avoid division by zero if a class is empty
  var1 = np.var(cls1) if len(cls1) > 0 else 0
  var2 = np.var(cls2) if len(cls2) > 0 else 0

  # Calculate weights based on the number of elements in each class
  weight1 = len(cls1) / len(valid_data)
  weight2 = len(cls2) / len(valid_data)

  return weight1 * var1 + weight2 * var2

# Load pre-compiled cohort dataset with all six features across all trials and sessions
excel_path = '/content/drive/MyDrive/<DATA_ROOT>/combined_feature_dataset.xlsx'
df = pd.read_excel(excel_path)
p1_min = df['min_p1']
p1_max = df['max_p1']


# Prepare data for Otsu's method, dropping NaN values
data_for_otsu = p1_min.dropna().values  # threshold determined by p1 dist min values due to binomial distribution

# Calculate min and max integer values for the threshold search range
# Ensure data_for_otsu is not empty
if len(data_for_otsu) > 0:
    min_val = int(np.floor(np.min(data_for_otsu)))
    max_val = int(np.ceil(np.max(data_for_otsu)))

    # Ensure min_val is less than max_val for range to work
    if min_val >= max_val:
        # If all values are the same or range is invalid, threshold might be the value itself
        otsu_threshold_p1 = min_val
    else:
        # Find the threshold that minimizes the intraclass variance
        otsu_threshold_p1 = min(
            range(min_val, max_val + 1), # +1 to include max_val in the range search
            key=lambda th: otsu_intraclass_variance(data_for_otsu, th),
        )
    print(f"Calculated Otsu Threshold: {otsu_threshold_p1}")
else:
    print("No valid data to calculate Otsu threshold.")
    otsu_threshold_p1 = None # No threshold if data is empty

# Histogram
p1_max.hist(bins = 60, edgecolor = 'black', color = 'orange', alpha = 0.5, label = 'Inner Eye Angle (max)')
p1_min.hist(bins = 70, edgecolor = 'black', alpha = 0.5, label = 'Inner Eye Angle (min)')
plt.xlabel('Inner Eye Angle (degrees)', fontsize = 16)
plt.ylabel('Frequency', fontsize = 16)
plt.tick_params(axis='both', labelsize=16)
plt.axvline(otsu_threshold_p1, color = 'red', label = f'Otsu threshold: {otsu_threshold_p1}')  # vertical line indicating threshold value
plt.legend()

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/p1_otsu_threshold.pdf', format='pdf', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/p1_otsu_threshold.png', format='png', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/p1_otsu_threshold.tiff', format='TIFF', bbox_inches = 'tight')

plt.show()

In [ ]:
# opto + ctr Inner Eye Angle Figure with SEM

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')
import scipy.stats as stats

# === CONFIGURATION ===
ctr_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID].csv']  # control subject trials
opto_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID].csv']  # opto subject trials

ctr_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']    # control manual scores
opto_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']   # opto manual scores

# === FUNCTIONS ===
# === ANGLE FUNCTION ===
def compute_single_angle(xa, ya, xb, yb, xc, yc):
    a = np.array([xa, ya]) - np.array([xb, yb])
    c = np.array([xc, yc]) - np.array([xb, yb])
    cos_theta = np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c))
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return np.degrees(np.arccos(cos_theta))

# === SORTER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')


# list storing accuracies for each individual trial/folder
ctr_accuracies_per_trial = []
opto_accuracies_per_trial = []

# loop going through each folder_path and its corresponding excel_path for CONTROL
for ctr_trial_indx, (ctr_current_folder_path, ctr_current_excel_path) in enumerate(zip(ctr_folder_path, ctr_excel_path)):

  print(f"\n--- Processing Trial {ctr_trial_indx + 1}: {ctr_current_folder_path} ---")

  # === READ GROUND TRUTH for current trial ===
  df_truth = pd.read_excel(ctr_current_excel_path, header=None)
  actual_full_list = df_truth[0].astype(int) # Renamed to actual_full_list for clarity and consistency with velocity cell


  # === PRECOMPUTE INNER EYE ANGLES FOR ALL CSVs ===
  all_dlc_files_in_folder = sorted( # Renamed for clarity and consistency
      [f for f in os.listdir(ctr_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  num_excel_entries = len(actual_full_list)
  num_dlc_files = len(all_dlc_files_in_folder)

  if num_excel_entries != num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {ctr_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(num_excel_entries, num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned # This `actual` is now the aligned one
  valid_mask = actual != 3 # `valid_mask` will now have the correct length
  actual_filtered = actual[valid_mask].reset_index(drop=True)


  ctr_angle_lookup = []  # Will hold Series of angles for each file

  # Loop over sorted_files_aligned, which is now length-matched with actual_aligned
  for filename in sorted_files_aligned: # Changed from csv_files
      file_path = os.path.join(ctr_current_folder_path, filename)
      try:
          df = pd.read_csv(file_path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()

          # Extract the necessary tracking points
          top_x, top_y = df['top_x'], df['top_y']
          bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
          p1_x, p1_y = df['p1_x'], df['p1_y'] # Corrected to p1 for inner eye angle

          inner_angles = []
          for i in range(len(df)):
              try:
                  # Vertex B is p1 (inner corner)
                  angle = compute_single_angle(
                      top_x.iloc[i], top_y.iloc[i],       # Point A
                      p1_x.iloc[i], p1_y.iloc[i],         # Point B (Vertex)
                      bottom_x.iloc[i], bottom_y.iloc[i]  # Point C
                  )
              except:
                  angle = np.nan
              inner_angles.append(angle)

          ctr_angle_lookup.append(pd.Series(inner_angles))

      except:
          ctr_angle_lookup.append(pd.Series([]))   # empty list on error

  # === LOOP OVER ANGLE THRESHOLDS (FAST) ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store current accuracies
  ctr_current_trial_accuracies = []

  thresholds = range(0, 181)
  for angle_thresh in thresholds:
      predictions = []
      for angle_series in ctr_angle_lookup:
          if angle_series.empty:
              predictions.append(0)
          else:
              blink = (angle_series < angle_thresh).any()
              predictions.append(1 if blink else 0)

      # Ensure predicted_filtered also uses a mask corresponding to the aligned data
      # The valid_mask.values will now have the same length as pd.Series(predictions)
      pred_filtered = pd.Series(predictions)[valid_mask.values].reset_index(drop=True)
      correct = (actual_filtered == pred_filtered) | ((actual_filtered == 2) & (pred_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_pred_blinks = pred_filtered.sum()

      ctr_current_trial_accuracies.append(accuracy)

      print(f"{angle_thresh}\t{accuracy:.2f}\t{total_pred_blinks}")

  ctr_accuracies_per_trial.append(ctr_current_trial_accuracies)



# loop going through each folder_path and its corresponding excel_path for OPTO
for opto_trial_indx, (opto_current_folder_path, opto_current_excel_path) in enumerate(zip(opto_folder_path, opto_excel_path)):

  print(f"\n--- Processing Trial {opto_trial_indx + 1}: {opto_current_folder_path} ---")

    # === READ GROUND TRUTH for current trial ===
  df_truth = pd.read_excel(opto_current_excel_path, header=None)
  actual_full_list = df_truth[0].astype(int) # Renamed to actual_full_list for clarity and consistency with velocity cell


  # === PRECOMPUTE INNER EYE ANGLES FOR ALL CSVs ===
  all_dlc_files_in_folder = sorted( # Renamed for clarity and consistency
      [f for f in os.listdir(opto_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  num_excel_entries = len(actual_full_list)
  num_dlc_files = len(all_dlc_files_in_folder)

  if num_excel_entries != num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {opto_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(num_excel_entries, num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned # This `actual` is now the aligned one
  valid_mask = actual != 3 # `valid_mask` will now have the correct length
  actual_filtered = actual[valid_mask].reset_index(drop=True)


  opto_angle_lookup = []  # Will hold Series of angles for each file

  # Loop over sorted_files_aligned, which is now length-matched with actual_aligned
  for filename in sorted_files_aligned: # Changed from csv_files
      file_path = os.path.join(opto_current_folder_path, filename)
      try:
          df = pd.read_csv(file_path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()

          # Extract the necessary tracking points
          top_x, top_y = df['top_x'], df['top_y']
          bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
          p1_x, p1_y = df['p1_x'], df['p1_y'] # Corrected to p1 for inner eye angle

          opto_inner_angles = []
          for i in range(len(df)):
              try:
                  # Vertex B is p1 (inner corner)
                  angle_val = compute_single_angle( # Renamed to angle_val to avoid confusion
                      top_x.iloc[i], top_y.iloc[i],       # Point A
                      p1_x.iloc[i], p1_y.iloc[i],         # Point B (Vertex)
                      bottom_x.iloc[i], bottom_y.iloc[i]  # Point C
                  )
              except:
                  angle_val = np.nan
              opto_inner_angles.append(angle_val) # Appending angle_val

          opto_angle_lookup.append(pd.Series(opto_inner_angles))

      except:
          opto_angle_lookup.append(pd.Series([]))   # empty list on error

  # === LOOP OVER ANGLE THRESHOLDS (FAST) ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store current accuracies
  opto_current_trial_accuracies = []

  thresholds = range(0, 181)
  for angle_thresh in thresholds:
      predictions = []
      for angle_series in opto_angle_lookup:
          if angle_series.empty:
              predictions.append(0)
          else:
              blink = (angle_series < angle_thresh).any()
              predictions.append(1 if blink else 0)

      # Ensure predicted_filtered also uses a mask corresponding to the aligned data
      # The valid_mask.values will now have the same length as pd.Series(predictions)
      pred_filtered = pd.Series(predictions)[valid_mask.values].reset_index(drop=True)
      correct = (actual_filtered == pred_filtered) | ((actual_filtered == 2) & (pred_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_pred_blinks = pred_filtered.sum()

      opto_current_trial_accuracies.append(accuracy)

      print(f"{angle_thresh}\t{accuracy:.2f}\t{total_pred_blinks}")

  opto_accuracies_per_trial.append(opto_current_trial_accuracies)

# Calculating avg accuracy across trials (moved outside the trial loop)
ctr_avg_accuracy = []
opto_avg_accuracy = []
ctr_sems_per_threshold = []
opto_sems_per_threshold = [] # New list to store SEM for each threshold

from itertools import zip_longest

# fillvalue=None lets us identify and skip missing data points
for column in zip_longest(*ctr_accuracies_per_trial, fillvalue=None):
    ctr_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if ctr_valid_numbers:
      ctr_avg_accuracy.append(sum(ctr_valid_numbers) / len(ctr_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      ctr_sems_per_threshold.append(stats.sem(ctr_valid_numbers))
    else:
      ctr_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      ctr_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers


for column in zip_longest(*opto_accuracies_per_trial, fillvalue=None):
    opto_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if opto_valid_numbers:
      opto_avg_accuracy.append(sum(opto_valid_numbers) / len(opto_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      opto_sems_per_threshold.append(stats.sem(opto_valid_numbers))
    else:
      opto_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      opto_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers

# Convert lists to numpy arrays for element-wise operations
ctr_avg_accuracy_np = np.array(ctr_avg_accuracy)
ctr_sems_np = np.array(ctr_sems_per_threshold)
opto_avg_accuracy_np = np.array(opto_avg_accuracy)
opto_sems_np = np.array(opto_sems_per_threshold)

max_ctr_accuracy = np.max(ctr_avg_accuracy_np)
max_ctr_accuracy_threshold_index = np.argmax(ctr_avg_accuracy_np)
max_ctr_sem = ctr_sems_np[max_ctr_accuracy_threshold_index]

max_opto_accuracy = np.max(opto_avg_accuracy_np)
max_opto_accuracy_threshold_index = np.argmax(opto_avg_accuracy_np)
max_opto_sem = opto_sems_np[max_opto_accuracy_threshold_index]

# Combined max avg accuracy of experimental and control
opto_ctr_accuracy = []   # avg accuracy at each index of opto and ctr lists
# max_opto_ctr_accuracy = np.max(opto_ctr_accuracy)

for indx in range(len(ctr_avg_accuracy_np)):
  if abs(ctr_avg_accuracy_np[indx] - opto_avg_accuracy_np[indx]) < 2:
    opto_ctr_accuracy.append((ctr_avg_accuracy_np[indx] + opto_avg_accuracy_np[indx])/2)

  else:
    opto_ctr_accuracy.append(0)

# index of maximum avg accuracy of opto and ctr combined that are within 2% of each other
max_opto_ctr_accuracy_threshold_index = np.argmax(opto_ctr_accuracy) # index of max avg avvuracy value of opto and ctr list tgt

# Generating avg across trials graph
fig, ax = plt.subplots(figsize = (10, 6))
ax.plot(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np, label='Control', color = '#b80d07') # Use numpy array for plotting
ax.plot(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np, label='Opto', color = '#026bdb')
ax.set_xlabel('Inner Eye Angle Threshold (degrees)', fontsize = 16)
ax.set_ylabel('Classification Accuracy (%)', fontsize = 16)
ax.set_title('Accuracy vs. Inner Eye Angle Threshold for Control and Opto', fontsize = 16, fontweight = 'bold')
ax.grid(True, linestyle='--', alpha=0.7)
ax.tick_params(axis = 'both', labelsize = 16)
# SEM
# Use numpy arrays for fill_between and add alpha for better visualization
ax.fill_between(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np - ctr_sems_np, ctr_avg_accuracy_np + ctr_sems_np, alpha=0.2, label = 'ctr_SEM', color = '#b80d07')
ax.fill_between(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np - opto_sems_np, opto_avg_accuracy_np + opto_sems_np, alpha=0.2, label = 'opto_SEM', color = '#026bdb')

ax.axvline(x = max_opto_ctr_accuracy_threshold_index, color = 'black', label = 'Validated threshold')
ax.axvline(x=otsu_threshold_p1, color = 'black', linestyle = '--', label = 'Otsu threshold')

ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 2)

ax.legend(bbox_to_anchor = (0.5, -0.15), loc = 'upper center', ncol=6)

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_v_p1_threshold.pdf', format='pdf', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_p1_threshold.tiff', format='TIFF', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_p1_threshold.png', format='png', bbox_inches='tight')

plt.show()

print(f"Control - Maximum Accuracy: {max_ctr_accuracy:.2f}% at threshold index {max_ctr_accuracy_threshold_index}, with SEM: {max_ctr_sem:.2f}")
print(f"Opto - Maximum Accuracy: {max_opto_accuracy:.2f}% at threshold index {max_opto_accuracy_threshold_index}, with SEM: {max_opto_sem:.2f}")
print(f"Control + Opto Threshold Index: {max_opto_ctr_accuracy_threshold_index}")
print(f" Control accuracy at overall threshold index: {ctr_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}")
print(f"Opto accuracy at overall threshold index: {opto_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}")

print(f'Otsu control avg accuracy: {ctr_avg_accuracy_np[otsu_threshold_p1]}')
print(f'Otsu opto avg accuracy: {opto_avg_accuracy_np[otsu_threshold_p1]}')

In [ ]:
# Outer Eye Angle threshold

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Otsu's method to determine threshold
def otsu_intraclass_variance(image, threshold):
  # Handle cases where `image` might contain NaNs or be empty for robust calculation
  valid_data = image[~np.isnan(image)]
  if len(valid_data) == 0:
      return np.inf # Return infinity if no valid data to prevent errors

  cls1 = valid_data[valid_data >= threshold]
  cls2 = valid_data[valid_data < threshold]

  # Avoid division by zero if a class is empty
  var1 = np.var(cls1) if len(cls1) > 0 else 0
  var2 = np.var(cls2) if len(cls2) > 0 else 0

  # Calculate weights based on the number of elements in each class
  weight1 = len(cls1) / len(valid_data)
  weight2 = len(cls2) / len(valid_data)

  return weight1 * var1 + weight2 * var2

# Load pre-compiled cohort dataset with all six features across all trials and sessions
excel_path = '/content/drive/MyDrive/<DATA_ROOT>/combined_feature_dataset.xlsx'
df = pd.read_excel(excel_path)
p4_min = df['min_p4']
p4_max = df['max_p4']


# Prepare data for Otsu's method, dropping NaN values
data_for_otsu = p4_min.dropna().values  # threshold determined by p4 dist min values due to binomial distribution

# Calculate min and max integer values for the threshold search range
# Ensure data_for_otsu is not empty
if len(data_for_otsu) > 0:
    min_val = int(np.floor(np.min(data_for_otsu)))
    max_val = int(np.ceil(np.max(data_for_otsu)))

    # Ensure min_val is less than max_val for range to work
    if min_val >= max_val:
        # If all values are the same or range is invalid, threshold might be the value itself
        otsu_threshold_p4 = min_val
    else:
        # Find the threshold that minimizes the intraclass variance
        otsu_threshold_p4 = min(
            range(min_val, max_val + 1), # +1 to include max_val in the range search
            key=lambda th: otsu_intraclass_variance(data_for_otsu, th),
        )
    print(f"Calculated Otsu Threshold: {otsu_threshold_p4}")
else:
    print("No valid data to calculate Otsu threshold.")
    otsu_threshold_p4 = None # No threshold if data is empty

# Histogram
p4_max.hist(bins = 60, edgecolor = 'black', color = 'orange', alpha = 0.5, label = 'Outer Eye Angle (max)')
p4_min.hist(bins = 70, edgecolor = 'black', alpha = 0.5, label = 'Outer Eye Angle (min)')
plt.xlabel('Outer Angle (Degrees)', fontsize = 16)
plt.ylabel('Frequency', fontsize = 16)
plt.tick_params(axis = 'both', labelsize = 16)
plt.axvline(otsu_threshold_p4, color = 'red', label = f'Otsu threshold: {otsu_threshold_p4}')  # vertical line indicating threshold value
plt.legend()

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/p4_otsu_threshold.pdf', format='pdf', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/p4_otsu_threshold.png', format='png', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/p4_otsu_threshold.tiff', format='TIFF', bbox_inches = 'tight')

plt.show()


In [ ]:
# opto + ctr Outer Eye Angle Figure with SEM

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')
import scipy.stats as stats

# === CONFIGURATION ===
ctr_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID].csv']  # control subject trials
opto_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID].csv']  # opto subject trials

ctr_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']    # control manual scores
opto_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']   # opto manual scores

# === FUNCTIONS ===
# === ANGLE FUNCTION ===
def compute_single_angle(xa, ya, xb, yb, xc, yc):
    a = np.array([xa, ya]) - np.array([xb, yb])
    c = np.array([xc, yc]) - np.array([xb, yb])
    cos_theta = np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c))
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return np.degrees(np.arccos(cos_theta))

# === SORTER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')


# list storing accuracies for each individual trial/folder
ctr_accuracies_per_trial = []
opto_accuracies_per_trial = []

# loop going through each folder_path and its corresponding excel_path for CONTROL
for ctr_trial_indx, (ctr_current_folder_path, ctr_current_excel_path) in enumerate(zip(ctr_folder_path, ctr_excel_path)):

  print(f"\n--- Processing Trial {ctr_trial_indx + 1}: {ctr_current_folder_path} ---")

  # === READ GROUND TRUTH for current trial ===
  df_truth = pd.read_excel(ctr_current_excel_path, header=None)
  actual_full_list = df_truth[0].astype(int) # Renamed to actual_full_list for clarity and consistency with velocity cell


  # === PRECOMPUTE OUTER EYE ANGLES FOR ALL CSVs ===
  all_dlc_files_in_folder = sorted( # Renamed for clarity and consistency
      [f for f in os.listdir(ctr_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  num_excel_entries = len(actual_full_list)
  num_dlc_files = len(all_dlc_files_in_folder)

  if num_excel_entries != num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {ctr_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(num_excel_entries, num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned # This `actual` is now the aligned one
  valid_mask = actual != 3 # `valid_mask` will now have the correct length
  actual_filtered = actual[valid_mask].reset_index(drop=True)


  ctr_angle_lookup = []  # Will hold Series of angles for each file

  # Loop over sorted_files_aligned, which is now length-matched with actual_aligned
  for filename in sorted_files_aligned: # Changed from csv_files
      file_path = os.path.join(ctr_current_folder_path, filename)
      try:
          df = pd.read_csv(file_path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()

          # Extract the necessary tracking points
          top_x, top_y = df['top_x'], df['top_y']
          bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
          p4_x, p4_y = df['p4_x'], df['p4_y'] # Corrected to p4 for outer eye angle

          outer_angles = []
          for i in range(len(df)):
              try:
                  # Vertex B is p4 (outer corner)
                  angle = compute_single_angle(
                      top_x.iloc[i], top_y.iloc[i],       # Point A
                      p4_x.iloc[i], p4_y.iloc[i],         # Point B (Vertex)
                      bottom_x.iloc[i], bottom_y.iloc[i]  # Point C
                  )
              except:
                  angle = np.nan
              outer_angles.append(angle)

          ctr_angle_lookup.append(pd.Series(outer_angles))

      except:
          ctr_angle_lookup.append(pd.Series([]))   # empty list on error

  # === LOOP OVER ANGLE THRESHOLDS (FAST) ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store current accuracies
  ctr_current_trial_accuracies = []

  thresholds = range(0, 181)
  for angle_thresh in thresholds:
      predictions = []
      for angle_series in ctr_angle_lookup:
          if angle_series.empty:
              predictions.append(0)
          else:
              blink = (angle_series < angle_thresh).any()
              predictions.append(1 if blink else 0)

      # Ensure predicted_filtered also uses a mask corresponding to the aligned data
      # The valid_mask.values will now have the same length as pd.Series(predictions)
      pred_filtered = pd.Series(predictions)[valid_mask.values].reset_index(drop=True)
      correct = (actual_filtered == pred_filtered) | ((actual_filtered == 2) & (pred_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_pred_blinks = pred_filtered.sum()

      ctr_current_trial_accuracies.append(accuracy)

      print(f"{angle_thresh}\t{accuracy:.2f}\t{total_pred_blinks}")

  ctr_accuracies_per_trial.append(ctr_current_trial_accuracies)



# loop going through each folder_path and its corresponding excel_path for OPTO
for opto_trial_indx, (opto_current_folder_path, opto_current_excel_path) in enumerate(zip(opto_folder_path, opto_excel_path)):

  print(f"\n--- Processing Trial {opto_trial_indx + 1}: {opto_current_folder_path} ---")

    # === READ GROUND TRUTH for current trial ===
  df_truth = pd.read_excel(opto_current_excel_path, header=None)
  actual_full_list = df_truth[0].astype(int) # Renamed to actual_full_list for clarity and consistency with velocity cell


  # === PRECOMPUTE OUTER EYE ANGLES FOR ALL CSVs ===
  all_dlc_files_in_folder = sorted( # Renamed for clarity and consistency
      [f for f in os.listdir(opto_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  num_excel_entries = len(actual_full_list)
  num_dlc_files = len(all_dlc_files_in_folder)

  if num_excel_entries != num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {opto_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(num_excel_entries, num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned # This `actual` is now the aligned one
  valid_mask = actual != 3 # `valid_mask` will now have the correct length
  actual_filtered = actual[valid_mask].reset_index(drop=True)


  opto_angle_lookup = []  # Will hold Series of angles for each file

  # Loop over sorted_files_aligned, which is now length-matched with actual_aligned
  for filename in sorted_files_aligned: # Changed from csv_files
      file_path = os.path.join(opto_current_folder_path, filename)
      try:
          df = pd.read_csv(file_path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()

          # Extract the necessary tracking points
          top_x, top_y = df['top_x'], df['top_y']
          bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
          p4_x, p4_y = df['p4_x'], df['p4_y']

          opto_outer_angles = []
          for i in range(len(df)):
              try:
                  # Vertex B is p4 (outer corner)
                  angle_val = compute_single_angle( # Renamed to angle_val to avoid confusion
                      top_x.iloc[i], top_y.iloc[i],       # Point A
                      p4_x.iloc[i], p4_y.iloc[i],         # Point B (Vertex)
                      bottom_x.iloc[i], bottom_y.iloc[i]  # Point C
                  )
              except:
                  angle_val = np.nan
              opto_outer_angles.append(angle_val) # Appending angle_val

          opto_angle_lookup.append(pd.Series(opto_outer_angles))

      except:
          opto_angle_lookup.append(pd.Series([]))   # empty list on error

  # === LOOP OVER ANGLE THRESHOLDS (FAST) ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  # Initialize list to store current accuracies
  opto_current_trial_accuracies = []

  thresholds = range(0, 181)
  for angle_thresh in thresholds:
      predictions = []
      for angle_series in opto_angle_lookup:
          if angle_series.empty:
              predictions.append(0)
          else:
              blink = (angle_series < angle_thresh).any()
              predictions.append(1 if blink else 0)

      # Ensure predicted_filtered also uses a mask corresponding to the aligned data
      # The valid_mask.values will now have the same length as pd.Series(predictions)
      pred_filtered = pd.Series(predictions)[valid_mask.values].reset_index(drop=True)
      correct = (actual_filtered == pred_filtered) | ((actual_filtered == 2) & (pred_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_pred_blinks = pred_filtered.sum()

      opto_current_trial_accuracies.append(accuracy)

      print(f"{angle_thresh}\t{accuracy:.2f}\t{total_pred_blinks}")

  opto_accuracies_per_trial.append(opto_current_trial_accuracies)

# Calculating avg accuracy across trials (moved outside the trial loop)
ctr_avg_accuracy = []
opto_avg_accuracy = []
ctr_sems_per_threshold = []
opto_sems_per_threshold = [] # New list to store SEM for each threshold

from itertools import zip_longest

# fillvalue=None lets us identify and skip missing data points
for column in zip_longest(*ctr_accuracies_per_trial, fillvalue=None):
    ctr_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if ctr_valid_numbers:
      ctr_avg_accuracy.append(sum(ctr_valid_numbers) / len(ctr_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      ctr_sems_per_threshold.append(stats.sem(ctr_valid_numbers))
    else:
      ctr_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      ctr_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers


for column in zip_longest(*opto_accuracies_per_trial, fillvalue=None):
    opto_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if opto_valid_numbers:
      opto_avg_accuracy.append(sum(opto_valid_numbers) / len(opto_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      opto_sems_per_threshold.append(stats.sem(opto_valid_numbers))
    else:
      opto_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      opto_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers

# Convert lists to numpy arrays for element-wise operations
ctr_avg_accuracy_np = np.array(ctr_avg_accuracy)
ctr_sems_np = np.array(ctr_sems_per_threshold)
opto_avg_accuracy_np = np.array(opto_avg_accuracy)
opto_sems_np = np.array(opto_sems_per_threshold)

max_ctr_accuracy = np.max(ctr_avg_accuracy_np)
max_ctr_accuracy_threshold_index = np.argmax(ctr_avg_accuracy_np)
max_ctr_sem = ctr_sems_np[max_ctr_accuracy_threshold_index]

max_opto_accuracy = np.max(opto_avg_accuracy_np)
max_opto_accuracy_threshold_index = np.argmax(opto_avg_accuracy_np)
max_opto_sem = opto_sems_np[max_opto_accuracy_threshold_index]

# Combined max avg accuracy of experimental and control
opto_ctr_accuracy = []   # avg accuracy at each index of opto and ctr lists
# max_opto_ctr_accuracy = np.max(opto_ctr_accuracy)

for indx in range(len(ctr_avg_accuracy_np)):
  if abs(ctr_avg_accuracy_np[indx] - opto_avg_accuracy_np[indx]) < 2:
    opto_ctr_accuracy.append((ctr_avg_accuracy_np[indx] + opto_avg_accuracy_np[indx])/2)

  else:
    opto_ctr_accuracy.append(0)

# index of maximum avg accuracy of opto and ctr combined that are within 2% of each other
max_opto_ctr_accuracy_threshold_index = np.argmax(opto_ctr_accuracy) # index of max avg avvuracy value of opto and ctr list tgt

# Generating avg across trials graph
fig, ax = plt.subplots(figsize = (10, 6))
ax.plot(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np, label='Control', color = '#b80d07') # Use numpy array for plotting
ax.plot(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np, label='Opto', color = '#026bdb')
ax.set_xlabel('Outer Eye Angle Threshold (degrees)', fontsize = 16)
ax.set_ylabel('Classification Accuracy (%)', fontsize = 16)
ax.set_title('Accuracy vs. Outer Eye Angle Threshold for Control and Opto', fontsize = 16, fontweight = 'bold')
ax.grid(True, linestyle='--', alpha=0.7)
ax.tick_params(axis = 'both', labelsize = 16)
# SEM
# Use numpy arrays for fill_between and add alpha for better visualization
ax.fill_between(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np - ctr_sems_np, ctr_avg_accuracy_np + ctr_sems_np, alpha=0.2, label = 'ctr_SEM', color = '#b80d07')
ax.fill_between(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np - opto_sems_np, opto_avg_accuracy_np + opto_sems_np, alpha=0.2, label = 'opto_SEM', color = '#026bdb')

ax.axvline(x = max_opto_ctr_accuracy_threshold_index, color = 'black', label = 'Validated threshold')
ax.axvline(x=otsu_threshold_p4, color = 'black', linestyle = '--', label = 'Otsu threshold')

ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 2)

ax.legend(bbox_to_anchor = (0.5, -0.15), loc = 'upper center', ncol=6)

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_v_p4_threshold.pdf', format='pdf', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_p4_threshold.tiff', format='TIFF', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_p4_threshold.png', format='png', bbox_inches = 'tight')

plt.show()

print(f"Control - Maximum Accuracy: {max_ctr_accuracy:.2f}% at threshold index {max_ctr_accuracy_threshold_index}, with SEM: {max_ctr_sem:.2f}")
print(f"Opto - Maximum Accuracy: {max_opto_accuracy:.2f}% at threshold index {max_opto_accuracy_threshold_index}, with SEM: {max_opto_sem:.2f}")
print(f"Control + Opto Threshold Index: {max_opto_ctr_accuracy_threshold_index}")
print(f'Control accuracy at overall threshold index: {ctr_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')
print(f'Opto accuracy at overall threshold index: {opto_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')

print(f'Otsu control avg accuracy: {ctr_avg_accuracy_np[otsu_threshold_p4]}')
print(f'Otsu opto avg accuracy: {opto_avg_accuracy_np[otsu_threshold_p4]}')

In [ ]:
# Velocity threshold

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Otsu's method to determine threshold
def otsu_intraclass_variance(image, threshold):
  # Handle cases where `image` might contain NaNs or be empty for robust calculation
  valid_data = image[~np.isnan(image)]
  if len(valid_data) == 0:
      return np.inf # Return infinity if no valid data to prevent errors

  cls1 = valid_data[valid_data >= threshold]
  cls2 = valid_data[valid_data < threshold]

  # Avoid division by zero if a class is empty
  var1 = np.var(cls1) if len(cls1) > 0 else 0
  var2 = np.var(cls2) if len(cls2) > 0 else 0

  # Calculate weights based on the number of elements in each class
  weight1 = len(cls1) / len(valid_data)
  weight2 = len(cls2) / len(valid_data)

  return weight1 * var1 + weight2 * var2

# Load pre-compiled cohort dataset with all six features across all trials and sessions
excel_path = '/content/drive/MyDrive/<DATA_ROOT>/combined_feature_dataset.xlsx'
df = pd.read_excel(excel_path)
velocity_min = df['min_velocity']
velocity_max = df['max_velocity']


# Prepare data for Otsu's method, dropping NaN values
data_for_otsu = velocity_max.dropna().values  # threshold determined by velocity max values due to binomial distribution

# Calculate min and max integer values for the threshold search range
# Ensure data_for_otsu is not empty
if len(data_for_otsu) > 0:
    min_val = int(np.floor(np.min(data_for_otsu)))
    max_val = int(np.ceil(np.max(data_for_otsu)))

    # Ensure min_val is less than max_val for range to work
    if min_val >= max_val:
        # If all values are the same or range is invalid, threshold might be the value itself
        otsu_threshold_vel = min_val
    else:
        # Find the threshold that minimizes the intraclass variance
        otsu_threshold_vel = min(
            range(min_val, max_val + 1), # +1 to include max_val in the range search
            key=lambda th: otsu_intraclass_variance(data_for_otsu, th),
        )
    print(f"Calculated Otsu Threshold: {otsu_threshold_vel}")
else:
    print("No valid data to calculate Otsu threshold.")
    otsu_threshold_vel = None # No threshold if data is empty

# Histogram
velocity_max.hist(bins = 60, edgecolor = 'black', color = 'orange', alpha = 0.5, label = 'Velocity (max)')
velocity_min.hist(bins = 70, edgecolor = 'black', alpha = 0.5, label = 'Velocity (min)')
plt.xlabel('Velocity (px/s)', fontsize = 16)
plt.ylabel('Frequency', fontsize = 16)
plt.tick_params(axis = 'both', labelsize = 16)
plt.axvline(otsu_threshold_vel, color = 'red', label = f'Otsu threshold: {otsu_threshold_vel}')  # vertical line indicating threshold value
plt.legend()

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/velocity_Otsu_threshold.pdf', format='pdf', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/velocity_otsu_threshold.png', format='png', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/velocity_otsu_threshold.tiff', format='TIFF', bbox_inches = 'tight')

plt.show()


In [ ]:
# opto + ctr Velocity Figure with SEM

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')
import scipy.stats as stats

# === CONFIGURATION ===
ctr_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID].csv']  # control subject trials
opto_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID].csv']  # opto subject trials

ctr_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']    # control manual scores
opto_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']   # opto manual scores

# === FUNCTIONS ===
fps = 30.0

# === FUNCTIONS ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

def compute_velocity_from_df(df):
    top_x, top_y = df['top_x'], df['top_y']
    bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
    eyelid_dist = np.sqrt((top_x - bottom_x)**2 + (top_y - bottom_y)**2)
    velocity = np.abs(np.diff(eyelid_dist)) * fps
    velocity = np.insert(velocity, 0, 0.0)
    return velocity

# list storing accuracies for each individual trial/folder

ctr_accuracies_per_trial = []
opto_accuracies_per_trial = []

# loop going through each folder_path and its corresponding excel_path for CONTROL
for ctr_trial_indx, (ctr_current_folder_path, ctr_current_excel_path) in enumerate(zip(ctr_folder_path, ctr_excel_path)):

  print(f"\n--- Processing Trial {ctr_trial_indx + 1}: {ctr_current_folder_path} ---")

   # === LOAD GROUND TRUTH ===
  df_gt = pd.read_excel(ctr_current_excel_path, header=None)
  actual_full_list = df_gt[0].astype(int)

  # === GET DLC FILES ===
  all_dlc_files_in_folder = sorted(
      [f for f in os.listdir(ctr_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  ctr_num_excel_entries = len(actual_full_list)
  ctr_num_dlc_files = len(all_dlc_files_in_folder)

  if ctr_num_excel_entries != ctr_num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {ctr_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(num_excel_entries, num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned
  valid_mask = actual != 3
  actual_filtered = actual[valid_mask].reset_index(drop=True)

  # === PRELOAD DATA ===
  ctr_all_velocity = []
  for filename in sorted_files_aligned:
      path = os.path.join(ctr_current_folder_path, filename)
      try:
          df = pd.read_csv(path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()
          ctr_velocity = compute_velocity_from_df(df)
          ctr_all_velocity.append((filename, ctr_velocity))
      except:
          ctr_all_velocity.append((filename, np.zeros(1)))  # fallback to 0 velocity

  # === SWEEP VELOCITY THRESHOLDS ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  ctr_current_trial_accuracies = [] # Initialize list to store accuracies of current trial

  # Define the range of thresholds for plotting
  threshold_values_for_plot = range(0, 751, 10)

  # Loop going through each threshold to predict blink classification and its accuracy
  for threshold in threshold_values_for_plot:
      predicted = []
      for _, velocity in ctr_all_velocity:
          blink = (velocity > threshold).any()
          predicted.append(1 if blink else 0)

      # Ensure predicted_filtered also uses a mask corresponding to the aligned data
      predicted_series = pd.Series(predicted)
      predicted_filtered = predicted_series[valid_mask.values].reset_index(drop=True)

      correct = (actual_filtered == predicted_filtered) | ((actual_filtered == 2) & (predicted_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_blinks = predicted_filtered.sum()

      ctr_current_trial_accuracies.append(accuracy)

      print(f"{threshold}\t{accuracy:.2f}\t{total_blinks}")

  ctr_accuracies_per_trial.append(ctr_current_trial_accuracies)



# loop going through each folder_path and its corresponding excel_path for OPTO
for opto_trial_indx, (opto_current_folder_path, opto_current_excel_path) in enumerate(zip(opto_folder_path, opto_excel_path)):

  print(f"\n--- Processing Trial {opto_trial_indx + 1}: {opto_current_folder_path} ---")

  # === LOAD GROUND TRUTH ===
  df_gt = pd.read_excel(opto_current_excel_path, header=None)
  actual_full_list = df_gt[0].astype(int)

  # === GET DLC FILES ===
  all_dlc_files_in_folder = sorted(
      [f for f in os.listdir(opto_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  opto_num_excel_entries = len(actual_full_list)
  opto_num_dlc_files = len(all_dlc_files_in_folder)

  if opto_num_excel_entries != opto_num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {opto_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(num_excel_entries, num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned
  valid_mask = actual != 3
  actual_filtered = actual[valid_mask].reset_index(drop=True)

  # === PRELOAD DATA ===
  opto_all_velocity = []
  for filename in sorted_files_aligned:
      path = os.path.join(opto_current_folder_path, filename)
      try:
          df = pd.read_csv(path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()
          opto_velocity = compute_velocity_from_df(df)
          opto_all_velocity.append((filename, opto_velocity))
      except:
          opto_all_velocity.append((filename, np.zeros(1)))  # fallback to 0 velocity

  # === SWEEP VELOCITY THRESHOLDS ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  opto_current_trial_accuracies = [] # Initialize list to store accuracies of current trial

  # Define the range of thresholds for plotting
  threshold_values_for_plot = range(0, 751, 10)

  # Loop going through each threshold to predict blink classification and its accuracy
  for threshold in threshold_values_for_plot:
      predicted = []
      for _, velocity in opto_all_velocity:
          blink = (velocity > threshold).any()
          predicted.append(1 if blink else 0)

      # Ensure predicted_filtered also uses a mask corresponding to the aligned data
      predicted_series = pd.Series(predicted)
      predicted_filtered = predicted_series[valid_mask.values].reset_index(drop=True)

      correct = (actual_filtered == predicted_filtered) | ((actual_filtered == 2) & (predicted_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_blinks = predicted_filtered.sum()

      opto_current_trial_accuracies.append(accuracy)

      print(f"{threshold}\t{accuracy:.2f}\t{total_blinks}")

  opto_accuracies_per_trial.append(opto_current_trial_accuracies)


# Calculating avg accuracy across trials (moved outside the trial loop)
ctr_avg_accuracy = []
opto_avg_accuracy = []
ctr_sems_per_threshold = []
opto_sems_per_threshold = [] # New list to store SEM for each threshold

from itertools import zip_longest

# fillvalue=None lets us identify and skip missing data points
for column in zip_longest(*ctr_accuracies_per_trial, fillvalue=None):
    ctr_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if ctr_valid_numbers:
      ctr_avg_accuracy.append(sum(ctr_valid_numbers) / len(ctr_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      ctr_sems_per_threshold.append(stats.sem(ctr_valid_numbers))
    else:
      ctr_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      ctr_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers


for column in zip_longest(*opto_accuracies_per_trial, fillvalue=None):
    opto_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if opto_valid_numbers:
      opto_avg_accuracy.append(sum(opto_valid_numbers) / len(opto_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      opto_sems_per_threshold.append(stats.sem(opto_valid_numbers))
    else:
      opto_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      opto_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers

# Convert lists to numpy arrays for element-wise operations
ctr_avg_accuracy_np = np.array(ctr_avg_accuracy)
ctr_sems_np = np.array(ctr_sems_per_threshold)
opto_avg_accuracy_np = np.array(opto_avg_accuracy)
opto_sems_np = np.array(opto_sems_per_threshold)

max_ctr_accuracy = np.max(ctr_avg_accuracy_np)
max_ctr_accuracy_threshold_index = np.argmax(ctr_avg_accuracy_np)
max_ctr_sem = ctr_sems_np[max_ctr_accuracy_threshold_index]

max_opto_accuracy = np.max(opto_avg_accuracy_np)
max_opto_accuracy_threshold_index = np.argmax(opto_avg_accuracy_np)
max_opto_sem = opto_sems_np[max_opto_accuracy_threshold_index]

# Combined max avg accuracy of experimental and control
opto_ctr_accuracy = []   # avg accuracy at each index of opto and ctr lists
# max_opto_ctr_accuracy = np.max(opto_ctr_accuracy)

for indx in range(len(ctr_avg_accuracy_np)):
  if abs(ctr_avg_accuracy_np[indx] - opto_avg_accuracy_np[indx]) < 2:
    opto_ctr_accuracy.append((ctr_avg_accuracy_np[indx] + opto_avg_accuracy_np[indx])/2)

  else:
    opto_ctr_accuracy.append(0)

# index of maximum avg accuracy of opto and ctr combined that are within 2% of each other
max_opto_ctr_accuracy_threshold_index = np.argmax(opto_ctr_accuracy) # index of max avg avvuracy value of opto and ctr list tgt


# Generating avg across trials graph
fig, ax = plt.subplots(figsize = (10, 6))
ax.plot(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np, label='Control', color = '#b80d07') # Use numpy array for plotting
ax.plot(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np, label='Opto', color = '#026bdb')
ax.set_xlabel('Velocity Threshold (px/s)', fontsize = 16)
ax.set_ylabel('Classification Accuracy (%)', fontsize = 16)
ax.set_title('Accuracy vs. Velocity Threshold for Control and Opto', fontsize = 16, fontweight = 'bold')
ax.grid(True, linestyle='--', alpha=0.7)
ax.tick_params(axis = 'both', labelsize = 16)
# SEM
# Use numpy arrays for fill_between and add alpha for better visualization
ax.fill_between(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np - ctr_sems_np, ctr_avg_accuracy_np + ctr_sems_np, alpha=0.2, label = 'ctr_SEM', color = '#b80d07')
ax.fill_between(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np - opto_sems_np, opto_avg_accuracy_np + opto_sems_np, alpha=0.2, label = 'opto_SEM', color = '#026bdb')

ax.axvline(x = max_opto_ctr_accuracy_threshold_index, color = 'black', label = 'Validated threshold')
ax.axvline(x=otsu_threshold_vel/10, color = 'black', linestyle = '--', label = 'Otsu threshold')

ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 5)

ax.legend(bbox_to_anchor = (0.5, -0.15), loc = 'upper center', ncol=6)

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_v_velocity_threshold.pdf', format='pdf', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_velocity_threshold.tiff', format='TIFF', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_velocity_threshold.png', format='png', bbox_inches = 'tight')

plt.show()

print(f"Control - Maximum Accuracy: {max_ctr_accuracy:.2f}% at threshold index {max_ctr_accuracy_threshold_index}, with SEM: {max_ctr_sem:.2f}")
print(f"Opto - Maximum Accuracy: {max_opto_accuracy:.2f}% at threshold index {max_opto_accuracy_threshold_index}, with SEM: {max_opto_sem:.2f}")
print(f"Control + Opto Threshold Index: {max_opto_ctr_accuracy_threshold_index}")
print(f'Control accuracy at overall threshold index: {ctr_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')
print(f'Opto accuracy at overall threshold index: {opto_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')

print(f'Otsu control avg accuracy: {ctr_avg_accuracy_np[round(otsu_threshold_vel/10)]}')
print(f'Otsu opto avg accuracy: {opto_avg_accuracy_np[round(otsu_threshold_vel/10)]}')

In [ ]:
# Eye Area threshold

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Otsu's method to determine threshold
def otsu_intraclass_variance(image, threshold):
  # Handle cases where `image` might contain NaNs or be empty for robust calculation
  valid_data = image[~np.isnan(image)]
  if len(valid_data) == 0:
      return np.inf # Return infinity if no valid data to prevent errors

  cls1 = valid_data[valid_data >= threshold]
  cls2 = valid_data[valid_data < threshold]

  # Avoid division by zero if a class is empty
  var1 = np.var(cls1) if len(cls1) > 0 else 0
  var2 = np.var(cls2) if len(cls2) > 0 else 0

  # Calculate weights based on the number of elements in each class
  weight1 = len(cls1) / len(valid_data)
  weight2 = len(cls2) / len(valid_data)

  return weight1 * var1 + weight2 * var2

# Load pre-compiled cohort dataset with all six features across all trials and sessions
excel_path = '/content/drive/MyDrive/<DATA_ROOT>/combined_feature_dataset.xlsx'
df = pd.read_excel(excel_path)
eye_area_min = df['min_eye_area']
eye_area_max = df['max_eye_area']


# Prepare data for Otsu's method, dropping NaN values
data_for_otsu = eye_area_min.dropna().values  # threshold determined by p1 dist min values due to binomial distribution

# Calculate min and max integer values for the threshold search range
# Ensure data_for_otsu is not empty
if len(data_for_otsu) > 0:
    min_val = int(np.floor(np.min(data_for_otsu)))
    max_val = int(np.ceil(np.max(data_for_otsu)))

    # Ensure min_val is less than max_val for range to work
    if min_val >= max_val:
        # If all values are the same or range is invalid, threshold might be the value itself
        otsu_threshold_area = min_val
    else:
        # Find the threshold that minimizes the intraclass variance
        otsu_threshold_area = min(
            range(min_val, max_val + 1), # +1 to include max_val in the range search
            key=lambda th: otsu_intraclass_variance(data_for_otsu, th),
        )
    print(f"Calculated Otsu Threshold: {otsu_threshold_area}")
else:
    print("No valid data to calculate Otsu threshold.")
    otsu_threshold_area = None # No threshold if data is empty

# Histogram
eye_area_max.hist(bins = 60, edgecolor = 'black', color = 'orange', alpha = 0.5, label = 'Eye Area (max)')
eye_area_min.hist(bins = 70, edgecolor = 'black', alpha = 0.5, label = 'Eye Area (min)')
plt.xlabel(r'Eye Area ($px^2$)', fontsize = 16)
plt.ylabel('Frequency', fontsize = 16)
plt.tick_params(axis='both', labelsize = 16)
plt.axvline(otsu_threshold_area, color = 'red', label = f'Otsu threshold: {otsu_threshold_area}')  # vertical line indicating threshold value
plt.legend()

plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/eye_area_otsu_threshold.pdf', format='pdf', bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/eye_area_otsu_threshold.png', format='png', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/eye_area_otsu_threshold.tiff', format='TIFF', bbox_inches = 'tight')

plt.show()


In [ ]:
# Opto + ctr Eye Area figure with SEM

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')
import scipy.stats as stats

# === CONFIGURATION ===
ctr_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID].csv']  # control subject trials
opto_folder_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID].csv',
                   '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID].csv']  # opto subject trials

ctr_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/ctr/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']    # control manual scores
opto_excel_path = ['/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT1_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT2_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT3_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT4_ID]_[SESSION_ID]_scores.xlsx',
                  '/content/drive/MyDrive/<DATA_ROOT>/opto/[SUBJECT5_ID]_[SESSION_ID]_scores.xlsx']   # opto manual scores

# === FUNCTIONS ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

def polygon_area(x, y):
    """Calculate the area of a polygon using the shoelace formula."""
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def compute_eye_area_from_df(df):
    keypoints = ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'top', 'bottom']
    x_coords = [df[f"{pt}_x"].values for pt in keypoints]
    y_coords = [df[f"{pt}_y"].values for pt in keypoints]
    x = np.stack(x_coords, axis=1)  # shape: (frames, 8)
    y = np.stack(y_coords, axis=1)
    return np.array([polygon_area(x[i], y[i]) for i in range(len(df))])

# list storing accuracies for each individual trial/folder

ctr_accuracies_per_trial = []
opto_accuracies_per_trial = []

# loop going through each folder_path and its corresponding excel_path for CONTROL
for ctr_trial_indx, (ctr_current_folder_path, ctr_current_excel_path) in enumerate(zip(ctr_folder_path, ctr_excel_path)):

  print(f"\n--- Processing Trial {ctr_trial_indx + 1}: {ctr_current_folder_path} ---")

   # === LOAD GROUND TRUTH ===
  df_gt = pd.read_excel(ctr_current_excel_path, header=None)
  actual_full_list = df_gt[0].astype(int)

  # === GET DLC FILES ===
  all_dlc_files_in_folder = sorted(
      [f for f in os.listdir(ctr_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  ctr_num_excel_entries = len(actual_full_list)
  ctr_num_dlc_files = len(all_dlc_files_in_folder)

  if ctr_num_excel_entries != ctr_num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {ctr_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(ctr_num_excel_entries, ctr_num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned
  valid_mask = actual != 3
  actual_filtered = actual[valid_mask].reset_index(drop=True)

  # === PRELOAD DATA (and save eye areas) ===
  ctr_all_eye_areas = []
  for filename in sorted_files_aligned: # Iterate over the ALIGNED list of files
      path = os.path.join(ctr_current_folder_path, filename)
      try:
          df = pd.read_csv(path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()
          eye_area = compute_eye_area_from_df(df)
          ctr_all_eye_areas.append((filename, eye_area))

      except Exception as e:
          print(f"Error processing {filename}: {e}")
          ctr_all_eye_areas.append((filename, np.full(1, 9999)))  # large fallback to prevent downstream errors

  # === SWEEP EYE AREA THRESHOLDS ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  ctr_current_trial_accuracies = [] # Initialize list to store accuracies of current trial

  # Define the range of thresholds for plotting
  threshold_values = range(0, 1501, 10)

  for threshold in threshold_values:
      predictions = []
      for filename, eye_area_data in ctr_all_eye_areas:
          if eye_area_data.size == 0: # Check if eye_area_data is empty due to error
              predictions.append(0) # Or handle as appropriate for your data; here, assume no blink
          else:
              blink = (eye_area_data < threshold).any()
              predictions.append(1 if blink else 0)

      pred_filtered = pd.Series(predictions)[valid_mask.values].reset_index(drop=True)
      correct = (actual_filtered == pred_filtered) | ((actual_filtered == 2) & (pred_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_pred_blinks = pred_filtered.sum()

      ctr_current_trial_accuracies.append(accuracy)

      print(f"{threshold}\t{accuracy:.2f}\t{total_pred_blinks}")

  ctr_accuracies_per_trial.append(ctr_current_trial_accuracies)



# loop going through each folder_path and its corresponding excel_path for OPTO
for opto_trial_indx, (opto_current_folder_path, opto_current_excel_path) in enumerate(zip(opto_folder_path, opto_excel_path)):

  print(f"\n--- Processing Trial {opto_trial_indx + 1}: {opto_current_folder_path} ---")

  # === LOAD GROUND TRUTH ===
  df_gt = pd.read_excel(opto_current_excel_path, header=None)
  actual_full_list = df_gt[0].astype(int)

  # === GET DLC FILES ===
  all_dlc_files_in_folder = sorted(
      [f for f in os.listdir(opto_current_folder_path) if f.endswith(".csv") and "DLC" in f],
      key=extract_number
  )

  # --- ALIGN LENGTHS OF GROUND TRUTH AND DLC FILES ---
  opto_num_excel_entries = len(actual_full_list)
  opto_num_dlc_files = len(all_dlc_files_in_folder)

  if opto_num_excel_entries != opto_num_dlc_files:
      print(f"Warning: Mismatch detected for Trial {opto_trial_indx+1}. "
            f"Excel entries: {num_excel_entries}, DLC files: {num_dlc_files}. "
            f"Truncating both to {min(num_excel_entries, num_dlc_files)} to align.")
      min_length = min(opto_num_excel_entries, opto_num_dlc_files)
      actual_aligned = actual_full_list.iloc[:min_length].reset_index(drop=True)
      sorted_files_aligned = all_dlc_files_in_folder[:min_length]
  else:
      actual_aligned = actual_full_list
      sorted_files_aligned = all_dlc_files_in_folder

  # Use the aligned lists for subsequent processing
  actual = actual_aligned
  valid_mask = actual != 3
  actual_filtered = actual[valid_mask].reset_index(drop=True)

  # === PRELOAD DATA (and save eye areas) ===
  opto_all_eye_areas = []
  for filename in sorted_files_aligned: # Iterate over the ALIGNED list of files
      path = os.path.join(opto_current_folder_path, filename)
      try:
          df = pd.read_csv(path, header=[1, 2], index_col=0)
          df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]
          df = df.dropna()
          eye_area = compute_eye_area_from_df(df)
          opto_all_eye_areas.append((filename, eye_area))

      except Exception as e:
          print(f"Error processing {filename}: {e}")
          opto_all_eye_areas.append((filename, np.full(1, 9999)))  # large fallback to prevent downstream errors

  # === SWEEP EYE AREA THRESHOLDS ===
  print("Threshold\tAccuracy (%)\tTotal Predicted Blinks")

  opto_current_trial_accuracies = [] # Initialize list to store accuracies of current trial

  # Define the range of thresholds for plotting
  threshold_values = range(0, 1501, 10)

  for threshold in threshold_values:
      predictions = []
      for filename, eye_area_data in opto_all_eye_areas:
          if eye_area_data.size == 0: # Check if eye_area_data is empty due to error
              predictions.append(0) # Or handle as appropriate for your data; here, assume no blink
          else:
              blink = (eye_area_data < threshold).any()
              predictions.append(1 if blink else 0)

      pred_filtered = pd.Series(predictions)[valid_mask.values].reset_index(drop=True)
      correct = (actual_filtered == pred_filtered) | ((actual_filtered == 2) & (pred_filtered == 0))
      accuracy = correct.sum() / len(correct) * 100
      total_pred_blinks = pred_filtered.sum()

      opto_current_trial_accuracies.append(accuracy)

      print(f"{threshold}\t{accuracy:.2f}\t{total_pred_blinks}")

  opto_accuracies_per_trial.append(opto_current_trial_accuracies)


# Calculating avg accuracy across trials (moved outside the trial loop)
ctr_avg_accuracy = []
opto_avg_accuracy = []
ctr_sems_per_threshold = []
opto_sems_per_threshold = [] # New list to store SEM for each threshold

from itertools import zip_longest

# fillvalue=None lets us identify and skip missing data points
for column in zip_longest(*ctr_accuracies_per_trial, fillvalue=None):
    ctr_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if ctr_valid_numbers:
      ctr_avg_accuracy.append(sum(ctr_valid_numbers) / len(ctr_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      ctr_sems_per_threshold.append(stats.sem(ctr_valid_numbers))
    else:
      ctr_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      ctr_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers


for column in zip_longest(*opto_accuracies_per_trial, fillvalue=None):
    opto_valid_numbers = [num for num in column if num is not None]  # Define valid_numbers
    if opto_valid_numbers:
      opto_avg_accuracy.append(sum(opto_valid_numbers) / len(opto_valid_numbers))
      # Calculate SEM for the current threshold's valid accuracies
      opto_sems_per_threshold.append(stats.sem(opto_valid_numbers))
    else:
      opto_avg_accuracy.append(0.0) # Append 0.0 or np.nan if there are no valid numbers
      opto_sems_per_threshold.append(0.0) # Append 0.0 for SEM if no valid numbers

# Convert lists to numpy arrays for element-wise operations
ctr_avg_accuracy_np = np.array(ctr_avg_accuracy)
ctr_sems_np = np.array(ctr_sems_per_threshold)
opto_avg_accuracy_np = np.array(opto_avg_accuracy)
opto_sems_np = np.array(opto_sems_per_threshold)

max_ctr_accuracy = np.max(ctr_avg_accuracy_np)   # finding the max average accuracy value from the control list
max_ctr_accuracy_threshold_index = np.argmax(ctr_avg_accuracy_np)   # determining the index of the max average ctr accuracy within the list
max_ctr_sem = ctr_sems_np[max_ctr_accuracy_threshold_index]

max_opto_accuracy = np.max(opto_avg_accuracy_np)
max_opto_accuracy_threshold_index = np.argmax(opto_avg_accuracy_np)
max_opto_sem = opto_sems_np[max_opto_accuracy_threshold_index]

# Combined max avg accuracy of experimental and control
opto_ctr_accuracy = []   # avg accuracy at each index of opto and ctr lists
# max_opto_ctr_accuracy = np.max(opto_ctr_accuracy)

for indx in range(len(ctr_avg_accuracy_np)):
  if abs(ctr_avg_accuracy_np[indx] - opto_avg_accuracy_np[indx]) < 2:
    opto_ctr_accuracy.append((ctr_avg_accuracy_np[indx] + opto_avg_accuracy_np[indx])/2)

  else:
    opto_ctr_accuracy.append(0)

# index of maximum avg accuracy of opto and ctr combined that are within 2% of each other
max_opto_ctr_accuracy_threshold_index = np.argmax(opto_ctr_accuracy) # index of max avg avvuracy value of opto and ctr list tgt

# Generating avg across trials graph
fig, ax = plt.subplots(figsize = (10,6))
ax.plot(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np, label='Control', color = '#b80d07') # Use numpy array for plotting
ax.plot(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np, label='Opto', color = '#026bdb')
ax.set_xlabel(r'Eye Area Threshold ($px^2$)', fontsize = 16)
ax.set_ylabel('Classification Accuracy (%)', fontsize = 16)
ax.set_title('Accuracy vs. Eye Area Threshold for Control and Opto', fontsize = 16, fontweight = 'bold')
ax.grid(True, linestyle='--', alpha=0.7)
ax.tick_params(axis='both', labelsize=16)

# SEM
# Use numpy arrays for fill_between and add alpha for better visualization
ax.fill_between(range(0, len(ctr_avg_accuracy_np)), ctr_avg_accuracy_np - ctr_sems_np, ctr_avg_accuracy_np + ctr_sems_np, alpha=0.2, label = 'ctr_SEM', color = '#b80d07')
ax.fill_between(range(0, len(opto_avg_accuracy_np)), opto_avg_accuracy_np - opto_sems_np, opto_avg_accuracy_np + opto_sems_np, alpha=0.2, label = 'opto_SEM', color = '#026bdb')

# Vertical line at threshold for max average accuracy of combined opto and ctr values

ax.axvline(x = max_opto_ctr_accuracy_threshold_index, color = 'black', label = 'Validated threshold')
ax.axvline(x =otsu_threshold_area/10 , color = 'black', linestyle = '--', label = 'Otsu threshold')


ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 2)

ax.legend(bbox_to_anchor = (0.5, -0.15), loc = 'upper center', ncol=6)


plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_v_eye_area_threshold.pdf', format='pdf', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_eye_area_threshold.tiff', format='TIFF', bbox_inches = 'tight')
plt.savefig('/content/drive/MyDrive/<DATA_ROOT>/accuracy_vs_eye_area_threshold.png', format='png', bbox_inches='tight')

plt.show()



print(f"Control - Maximum Accuracy: {max_ctr_accuracy:.2f}% at threshold index {max_ctr_accuracy_threshold_index}, with SEM: {max_ctr_sem:.2f}")
print(f"Opto - Maximum Accuracy: {max_opto_accuracy:.2f}% at threshold index {max_opto_accuracy_threshold_index}, with SEM: {max_opto_sem:.2f}")
print(f"Control + Opto Threshold Index: {max_opto_ctr_accuracy_threshold_index}")
print(f'Control accuracy at overall threshold index: {ctr_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')
print(f'Opto accuracy at overall threshold index: {opto_avg_accuracy_np[max_opto_ctr_accuracy_threshold_index]}')

print(f'Otsu control avg accuracy: {ctr_avg_accuracy_np[round(otsu_threshold_area/10)]}')
print(f'Otsu opto avg accuracy: {opto_avg_accuracy_np[round(otsu_threshold_area/10)]}')